In [1]:
import os
import pyvista as pv
import matplotlib.pyplot as plt
from glob import glob
from natsort import natsorted
os.environ['PYVISTA_OFF_SCREEN'] = 'true'
pv.set_jupyter_backend('client')# pv.start_xvfb()                              
pv.global_theme.trame.server_proxy_enabled = True
pv.global_theme.trame.server_proxy_prefix = "/proxy/"

ModuleNotFoundError: No module named 'natsort'

In [ ]:
base_pth = "./128"
out_pth = os.path.join(base_pth, "output_drainage")
out_sim = glob(out_pth + "/*.vtk")
out_sim = natsorted(out_sim)
print(len(out_sim))

In [ ]:
idx = 25
print(out_sim[idx])
sample = pv.read(out_sim[idx])
print(sample)

In [ ]:
print(sample.array_names)        # list all 17 field names 
print(sample['rho_water'].shape)

In [ ]:
def get_phase(sample, nwp_key, wp_key, target="nwp"):
    nx, ny, nz = sample.dimensions
    nwp = sample[nwp_key].reshape(nx-1, ny-1, nz-1, order='F')
    wp = sample[wp_key].reshape(nx-1, ny-1, nz-1, order='F')
    out = nwp > wp if target == "nwp" else nwp < wp
    return out

def get_rock(sample):
    nx, ny, nz = sample.dimensions
    out = sample['flag'].reshape(nx-1, ny-1, nz-1, order='F')
    return out

In [ ]:
def plot_slice(sample, z=100):
    nx, ny, nz = sample.dimensions
    rho_air  = sample["rho_air"].reshape(nx-1, ny-1, nz-1, order="F")
    rho_water = sample["rho_water"].reshape(nx-1, ny-1, nz-1, order="F")
    phase = get_phase(sample, "rho_air", "rho_water", target="nwp")
    rock  = get_rock(sample)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    titles = ["Phase (NWP threshold)", "rho_air", "rho_water", "Rock"]
    data   = [phase[:, :, z], rho_air[:, :, z], rho_water[:, :, z], rock[:, :, z]]

    for ax, d, title in zip(axes, data, titles):
        im = ax.imshow(d, origin="lower")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
for i in range(50):
    idx = i
    print(f"plotting for sample at {out_sim[idx]}")
    sample = pv.read(out_sim[idx])
    plot_slice(sample, z=100)